In [61]:
from dotenv import load_dotenv
import os
from polygon import RESTClient
import json
import requests
import pydantic
import dataclasses
import inspect
import pandas as pd
import plotly.express as px
from datetime import datetime
import pytz
load_dotenv()
POLYGON_API_KEY = os.getenv("POLYGON_API_KEY")

In [2]:
client = RESTClient(POLYGON_API_KEY)


In [41]:
intl_daily_agg = client.get_daily_open_close_agg("AAPL","2025-09-03")

In [42]:
intl_daily_agg

DailyOpenCloseAgg(after_hours=238.1392, close=238.47, from_='2025-09-03', high=238.85, low=234.36, open=237.21, pre_market=235.85, status='OK', symbol='AAPL', volume=66427835.0, otc=None)

In [43]:
intl_daily_agg_5min = client.get_aggs("AAPL",5,"minute","2025-09-03","2025-09-03")

In [44]:
intl_daily_agg_5min

[Agg(open=235.85, high=235.85, low=235.43, close=235.78, volume=12828, vwap=235.6427, timestamp=1756886400000, transactions=602, otc=None),
 Agg(open=235.7, high=235.7, low=235.52, close=235.69, volume=8186, vwap=235.6329, timestamp=1756886700000, transactions=281, otc=None),
 Agg(open=235.62, high=235.9, low=235.62, close=235.88, volume=9313, vwap=235.7415, timestamp=1756887000000, transactions=314, otc=None),
 Agg(open=235.92, high=235.94, low=235.84, close=235.94, volume=4565, vwap=235.8871, timestamp=1756887300000, transactions=141, otc=None),
 Agg(open=236, high=236.2, low=235.99, close=236.15, volume=11075, vwap=236.0553, timestamp=1756887600000, transactions=365, otc=None),
 Agg(open=236.1, high=236.15, low=236.02, close=236.02, volume=3861, vwap=236.0964, timestamp=1756887900000, transactions=44, otc=None),
 Agg(open=236.04, high=236.15, low=236.03, close=236.04, volume=7382, vwap=236.0733, timestamp=1756888200000, transactions=145, otc=None),
 Agg(open=236.04, high=236.13, low

In [45]:
type(intl_daily_agg_5min[0])

polygon.rest.models.aggs.Agg

In [46]:
agg = intl_daily_agg_5min[0]
print(dataclasses.is_dataclass(agg))
print(getattr(agg,"__slots__",None))
print([n for n in dir(agg) if not n.startswith("_")])
dataclasses.asdict(agg)

True
None
['close', 'from_dict', 'high', 'low', 'open', 'otc', 'timestamp', 'transactions', 'volume', 'vwap']


{'open': 235.85,
 'high': 235.85,
 'low': 235.43,
 'close': 235.78,
 'volume': 12828,
 'vwap': 235.6427,
 'timestamp': 1756886400000,
 'transactions': 602,
 'otc': None}

In [47]:
intl_daily_agg_5min_dicts = [dataclasses.asdict(agg) for agg in intl_daily_agg_5min]
intl_daily_agg_5min_dicts

[{'open': 235.85,
  'high': 235.85,
  'low': 235.43,
  'close': 235.78,
  'volume': 12828,
  'vwap': 235.6427,
  'timestamp': 1756886400000,
  'transactions': 602,
  'otc': None},
 {'open': 235.7,
  'high': 235.7,
  'low': 235.52,
  'close': 235.69,
  'volume': 8186,
  'vwap': 235.6329,
  'timestamp': 1756886700000,
  'transactions': 281,
  'otc': None},
 {'open': 235.62,
  'high': 235.9,
  'low': 235.62,
  'close': 235.88,
  'volume': 9313,
  'vwap': 235.7415,
  'timestamp': 1756887000000,
  'transactions': 314,
  'otc': None},
 {'open': 235.92,
  'high': 235.94,
  'low': 235.84,
  'close': 235.94,
  'volume': 4565,
  'vwap': 235.8871,
  'timestamp': 1756887300000,
  'transactions': 141,
  'otc': None},
 {'open': 236,
  'high': 236.2,
  'low': 235.99,
  'close': 236.15,
  'volume': 11075,
  'vwap': 236.0553,
  'timestamp': 1756887600000,
  'transactions': 365,
  'otc': None},
 {'open': 236.1,
  'high': 236.15,
  'low': 236.02,
  'close': 236.02,
  'volume': 3861,
  'vwap': 236.0964,
 

In [48]:
df = pd.DataFrame(intl_daily_agg_5min_dicts)
df["timestamp_utc"] = pd.to_datetime(df["timestamp"],unit="ms",utc=True)
df["timestamp_et"] = df["timestamp_utc"].dt.tz_convert("America/New_York")
df

,open,high,low,close,volume,vwap,timestamp,transactions,otc,timestamp_utc,timestamp_et
0,235.85,235.8500,235.4300,235.7800,12828.0,235.6427,1756886400000,602,None,2025-09-03 08:00:00+00:00,2025-09-03 04:00:00-04:00
1,235.70,235.7000,235.5200,235.6900,8186.0,235.6329,1756886700000,281,None,2025-09-03 08:05:00+00:00,2025-09-03 04:05:00-04:00
2,235.62,235.9000,235.6200,235.8800,9313.0,235.7415,1756887000000,314,None,2025-09-03 08:10:00+00:00,2025-09-03 04:10:00-04:00
3,235.92,235.9400,235.8400,235.9400,4565.0,235.8871,1756887300000,141,None,2025-09-03 08:15:00+00:00,2025-09-03 04:15:00-04:00
4,236.00,236.2000,235.9900,236.1500,11075.0,236.0553,1756887600000,365,None,2025-09-03 08:20:00+00:00,2025-09-03 04:20:00-04:00
...,...,...,...,...,...,...,...,...,...,...,...
187,238.06,238.0700,238.0600,238.0606,1134.0,238.0649,1756942500000,50,None,2025-09-03 23:35:00+00:00,2025-09-03 19:35:00-04:00
188,238.14,238.1400,237.9900,238.0000,5897.0,238.0513,1756942800000,81,None,2025-09-03 23:40:00+00:00,2025-09-03 19:40:00-04:00
189,238.00,238.0800,237.9106,238.0800,5071.0,238.0082,1756943100000,83,None,2025-09-03 23:45:00+00:00,2025-09-03 19:45:00-04:00
190,238.06,238.1600,238.0600,238.0600,4883.0,238.1117,1756943400000,68,None,2025-09-03 23:50:00+00:00,2025-09-03 19:50:00-04:00


In [63]:
#PUlling 5 min aggregates for a 15 minute window
est = pytz.timezone("US/Eastern")
window_start = est.localize(datetime.strptime('2025-09-03 09:00:00',"%Y-%m-%d %H:%M:%S"))
window_end = est.localize(datetime.strptime('2025-09-03 09:15:00',"%Y-%m-%d %H:%M:%S"))
intl_5min_agg_over_15_mins = client.get_aggs("AAPL",5,"minute",window_start,window_end)
intl_5min_agg_over_15_mins_dict = [dataclasses.asdict(agg) for agg in intl_5min_agg_over_15_mins]
df15 = pd.DataFrame(intl_5min_agg_over_15_mins_dict)
df15["timestamp_utc"] = pd.to_datetime(df15["timestamp"],unit="ms",utc=True)
df15["timestamp_et"] = df15["timestamp_utc"].dt.tz_convert("America/New_York")
df15

,open,high,low,close,volume,vwap,timestamp,transactions,otc,timestamp_utc,timestamp_et
0,237.8600,238.3000,237.7000,238.2900,78062,237.9604,1756904400000,1273,None,2025-09-03 13:00:00+00:00,2025-09-03 09:00:00-04:00
1,238.2700,238.7500,238.1502,238.2514,131465,238.4658,1756904700000,1396,None,2025-09-03 13:05:00+00:00,2025-09-03 09:05:00-04:00
2,238.2545,238.2991,237.6601,237.8000,46115,237.9372,1756905000000,1050,None,2025-09-03 13:10:00+00:00,2025-09-03 09:10:00-04:00
3,237.7000,237.7615,237.2600,237.4000,60088,237.5413,1756905300000,1134,None,2025-09-03 13:15:00+00:00,2025-09-03 09:15:00-04:00


In [51]:
px.line(data_frame=df,x="timestamp_et",y="close")

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': 'timestamp_et=%{x}<br>close=%{y}<extra></extra>',
              'legendgroup': '',
              'line': {'color': '#636efa', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines',
              'name': '',
              'orientation': 'v',
              'showlegend': False,
              'type': 'scatter',
              'x': array([Timestamp('2025-09-03 04:00:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 04:05:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 04:10:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 04:15:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 04:20:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 04:25:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 04:30:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 04:35:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 04:40:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 04:45:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 04:50:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 04:55:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 05:00:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 05:05:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 05:10:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 05:15:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 05:20:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 05:25:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 05:30:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 05:35:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 05:40:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 05:45:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 05:50:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 05:55:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 06:00:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 06:05:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 06:10:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 06:15:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 06:20:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 06:25:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 06:30:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 06:35:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 06:40:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 06:45:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 06:50:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 06:55:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 07:00:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 07:05:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 07:10:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 07:15:00-0400', tz='America/New_York'),
                          Timestamp('2025-09-03 07:20:00-0400', tz='America/New_York'),
   